# Unsupervised: Generative Art Generation

Generate AI art using Stable Diffusion, influenced by weather parameters.

This notebook creates images that will be used in the Unsupervised art installation.

## Setup and Installation

In [ ]:
# Install required packages
!pip install -q diffusers transformers torch pillow accelerate omegaconf xformers

In [ ]:
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline, DDPMScheduler
from datetime import datetime
import os
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Weather Parameters (Set these based on desired conditions)

In [ ]:
import math
from datetime import datetime

def generate_environmental_factors():
    """
    Generate environmental factors without API keys.
    Uses: time patterns, deterministic randomization, and procedural generation.
    """
    now = datetime.now()
    hour = now.hour
    minute = now.minute
    second = now.second
    
    # Day of year for seasonal variations
    day_of_year = now.timetuple().tm_yday
    
    # Pseudo-random function (deterministic but appears random)
    def pseudo_random(seed):
        return abs(math.sin(seed * 12.9898) * math.sin(seed * 78.233)) % 1
    
    # Time-based seed
    time_seed = hour * 3600 + minute * 60 + second
    
    # Fractions for cycling patterns
    hour_fraction = hour / 24
    minute_fraction = minute / 60
    day_fraction = day_of_year / 365
    
    # Screen/device seed for variety
    device_seed = (1920 + 1080 + 2) / 1000  # Use fixed values for reproducibility
    
    # Temperature: cycles through day with randomization
    base_temp_cycle = math.sin((hour_fraction - 0.25) * math.pi * 2) * 15
    seasonal_temp = math.sin(day_fraction * math.pi * 2) * 10
    random_temp = (pseudo_random(time_seed + device_seed) - 0.5) * 20
    temperature = 20 + base_temp_cycle + seasonal_temp + random_temp
    
    # Humidity: creates wave-like patterns
    base_humidity = (math.sin(minute_fraction * math.pi * 2) * 0.3 + 0.5) * 100
    random_humidity = pseudo_random(time_seed * 1.5 + device_seed) * 40 - 20
    humidity = max(10, min(100, base_humidity + random_humidity))
    
    # Wind speed: peaks mid-day, calm at night
    base_wind = (math.sin((hour_fraction - 0.25) * math.pi) * 0.5 + 0.5) * 15
    random_wind = pseudo_random(time_seed * 2 + device_seed) * 10 - 5
    wind_speed = max(0, base_wind + random_wind)
    
    # Condition based on combination of factors
    if temperature > 30:
        temp_desc = "scorching"
    elif temperature > 20:
        temp_desc = "warm"
    elif temperature > 10:
        temp_desc = "cool"
    else:
        temp_desc = "cold"
    
    if humidity > 70:
        humidity_desc = "humid"
    elif humidity > 40:
        humidity_desc = "balanced"
    else:
        humidity_desc = "dry"
    
    if wind_speed > 15:
        wind_desc = "turbulent"
    elif wind_speed > 7:
        wind_desc = "flowing"
    else:
        wind_desc = "calm"
    
    condition = f"{temp_desc}, {humidity_desc}, {wind_desc}"
    
    return {
        'temperature': temperature,
        'humidity': humidity,
        'wind_speed': wind_speed,
        'condition': condition,
    }

# Generate environmental factors (no API needed!)
weather_params = generate_environmental_factors()

# Display current environmental scenario
print("=" * 50)
print("GENERATED ENVIRONMENTAL FACTORS")
print("=" * 50)
print(f"Temperature: {weather_params['temperature']:.1f}°C")
print(f"Humidity: {weather_params['humidity']:.0f}%")
print(f"Wind Speed: {weather_params['wind_speed']:.1f} m/s")
print(f"Condition: {weather_params['condition']}")
print("=" * 50)

## Generate Prompts Based on Weather

In [ ]:
def weather_to_visual_description(weather):
    """Convert weather parameters to artistic descriptions."""
    
    temp = weather['temperature']
    humidity = weather['humidity']
    wind = weather['wind_speed']
    condition = weather['condition']
    
    # Color palette based on temperature
    if temp < 5:
        color_desc = "icy blues, frost-like crystalline structures, cold ethereal light"
    elif temp < 15:
        color_desc = "cool blues and purples, misty atmosphere, calm flowing energy"
    elif temp < 25:
        color_desc = "balanced greens and teals, natural flowing patterns"
    else:
        color_desc = "warm reds, oranges, yellows, energetic and vibrant flows"
    
    # Density based on humidity
    if humidity < 30:
        density_desc = "sparse, minimal, breathing space, open geometry"
    elif humidity < 60:
        density_desc = "balanced composition, moderate complexity"
    else:
        density_desc = "dense, intricate patterns, layered complexity, flowing abundantly"
    
    # Motion based on wind
    if wind < 5:
        motion_desc = "serene, still, meditative calm, gentle movements"
    elif wind < 15:
        motion_desc = "flowing, graceful movements, dynamic elegance"
    else:
        motion_desc = "chaotic, turbulent, swirling forces, rapid transformations"
    
    # Overall aesthetic
    if condition == 'stormy':
        aesthetic = "dramatic, intense, turbulent, atmospheric disturbance"
    elif condition == 'rainy':
        aesthetic = "wet, reflective, layered, emotional depth"
    elif condition == 'cloudy':
        aesthetic = "diffused, soft, contemplative, transitional"
    else:  # clear
        aesthetic = "clear, bright, crystalline, pure geometry"
    
    full_description = f"{color_desc}, {density_desc}, {motion_desc}, {aesthetic}"
    return full_description

# Generate the prompt
visual_description = weather_to_visual_description(weather_params)
base_prompt = "abstract generative art, flowing particles, organic geometry, AI-generated"
full_prompt = f"{base_prompt}, {visual_description}"

print("\nGENERATED PROMPT:")
print("=" * 70)
print(full_prompt)
print("=" * 70)

## Load Stable Diffusion Model

In [ ]:
# Load model (first time will download ~5GB)
model_id = "stabilityai/stable-diffusion-2-1"
print(f"Loading model: {model_id}...")

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None  # Disable safety checker for art
)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()  # Saves memory

print("✓ Model loaded successfully!")

## Generate Images

In [ ]:
# Generation parameters
num_images = 4
image_height = 512
image_width = 512
num_inference_steps = 50  # Higher = better quality, slower
guidance_scale = 7.5      # Higher = more adherence to prompt

print(f"Generating {num_images} images...")
print(f"Resolution: {image_width}x{image_height}")
print(f"Steps: {num_inference_steps}")
print(f"Guidance scale: {guidance_scale}")
print("\nThis may take 2-5 minutes...\n")

# Generate with different seeds for variety
images = []
for i in range(num_images):
    seed = i + int(datetime.now().timestamp())  # Unique seed
    generator = torch.Generator("cuda").manual_seed(seed)
    
    with torch.autocast("cuda"):
        image = pipe(
            full_prompt,
            height=image_height,
            width=image_width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
        ).images[0]
    
    images.append(image)
    print(f"✓ Image {i+1}/{num_images} generated (seed: {seed})")

print("\nAll images generated!")

## Display Generated Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, (ax, img) in enumerate(zip(axes.flat, images)):
    ax.imshow(img)
    ax.set_title(f"Generated Image {idx+1}")
    ax.axis('off')

plt.tight_layout()
plt.savefig('generated_preview.png', dpi=150, bbox_inches='tight')
plt.show()

print("Preview saved as 'generated_preview.png'")

## Export Images

In [ ]:
# Create output directory
output_dir = Path('generated_art')
output_dir.mkdir(exist_ok=True)

# Save images
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
saved_paths = []

for idx, image in enumerate(images):
    filename = f"unsupervised_{timestamp}_{idx:03d}.png"
    filepath = output_dir / filename
    image.save(filepath)
    saved_paths.append(str(filepath))
    print(f"✓ Saved: {filename}")

# Save metadata
metadata = {
    'timestamp': timestamp,
    'weather_params': weather_params,
    'prompt': full_prompt,
    'model': model_id,
    'num_images': num_images,
    'resolution': f"{image_width}x{image_height}",
    'steps': num_inference_steps,
    'guidance_scale': guidance_scale,
    'images': saved_paths,
}

metadata_path = output_dir / f"metadata_{timestamp}.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata saved: {metadata_path}")
print(f"\n✓ All {num_images} images exported to '{output_dir}/'")

## Download Generated Images

In [ ]:
# Zip the generated images
import shutil

zip_path = 'generated_art.zip'
shutil.make_archive('generated_art', 'zip', 'generated_art')

from google.colab import files
files.download(zip_path)

print(f"✓ Downloaded: {zip_path}")
print("\nInstructions:")
print("1. Extract the ZIP file")
print("2. Copy images to: frontend/assets/generated/")
print("3. Update the visualization to display these images")